# 0. Setup

In [3]:
import ibis
import pandas as pd
from utils.f_0_dirs import get_data_dirs

dirs = get_data_dirs(segment="model")
con = ibis.duckdb.connect(dirs.db_path)

In [4]:
%%script true
from ibis import _, selectors as s
import pandas as pd

# 1. Execute the native Ibis aggregation (returns a single wide row)
t_panel = con.table("working_yearly_with_tfp_wave")
nb_rows = t_panel.count().execute()
raw_result = (
    t_panel
    .select(
        'tfp',
        'peer_tfp_ttwa_donut', 'peer_tfp_pc4_donut', 'peer_tfp_pc8',
        'tfp_wav1', 'tfp_wav2', 'tfp_wav3', 'nb_peers'
    )
    .aggregate(
        s.across(
            s.all(),
            {
                "count": _.count(),
                "percent": _.count() / nb_rows,
                "mean": _.mean(),
                "median": _.median(),
                "min": _.min(),
                "max": _.max(),
                "std": _.std(),
            }
        )
    )
    .execute()
)

# 2. Reshape the single row into a MultiIndex Series, then unstack
s_flat = raw_result.iloc[0]
s_flat.index = pd.MultiIndex.from_tuples(
    [col.rsplit("_", 1) for col in s_flat.index], 
    names=["variable", "metric"]
)

# 3. Pivot the metrics into separate columns
df_count = (
    s_flat
    .unstack(level="metric")[["count", "percent", "mean", "median", "min", "max", "std"]]
)

display(df_count.style.format({
    "count": "{:,.0f}",
    "percent": "{:.1%}",
    "mean": "{:,.3f}",
    "median": "{:,.3f}",
    "min": "{:,.3f}",
    "max": "{:,.3f}",
    "std": "{:,.3f}"
}))

Couldn't find program: 'true'


In [5]:
import ibis
from dataclasses import dataclass, field

from ibis import _
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PanelOLS

@dataclass
class ModelSpec(dict):
    Y: str
    X: list[str]
    fe: list[str]
    description: str
    # Use field(default_factory=list) for mutable defaults like arrays
    W_it: list[str] = field(default_factory=list)
    w_i: list[str] = field(default_factory=list)
    w_it: list[str] = field(default_factory=list)
    include: bool = True

def run_panel(mod: ModelSpec, t_panel: ibis.Table, model_name: str):
    
    # Mutate to dynamically log-transform all columns, adding ln_ prefix to column name
    log_keys = ['W_it']
    lvl_keys = ['Y', 'X', 'w_i', 'w_it']
    log_raw_params: list[str] = []
    lvl_raw_params: list[str] = []
    for key in log_keys:
        value = getattr(mod, key)
        if isinstance(value, list):
            log_raw_params.extend(value)
        else:
            log_raw_params.append(value)
    for key in lvl_keys:
        value = getattr(mod, key)
        if isinstance(value, list):
            lvl_raw_params.extend(value)
        else:
            lvl_raw_params.append(value)
    full_params = (
        ['registered_number', 'year'] +
        [f'ln_{p}' for p in log_raw_params] +
        [f'{p}' for p in lvl_raw_params]
    )
    if len(log_raw_params) > 0:
        table_logged = (
            t_panel
            .filter(_[p] > 0 for p in log_raw_params)
            .mutate(**{f'{p}': np.log(_[p]) for p in log_raw_params})
            .rename({ f'ln_{p}': f'{p}' for p in log_raw_params })
        )
    else:
        table_logged = t_panel
    table_logged = table_logged.select(full_params)
    if 'Y' in log_keys:
        table_start = table_logged.rename({ 'ln_Y': f'ln_{mod.Y}' })
    else:
        table_start = table_logged.rename({ 'Y': f'{mod.Y}' })
    table_start = table_start.execute()

    # 1. Execute into a Pandas DataFrame and set the MultiIndex for linearmodels
    df_model = table_start.set_index(['registered_number', 'year'])
    regressor_params = [p for p in full_params if p not in ['registered_number', 'year', f'ln_{mod.Y}', mod.Y]]

    # Define Endogenous (Y) and Exogenous (X) variables
    Y = df_model['ln_Y'] if 'ln_Y' in df_model.columns else df_model['Y']
    X = sm.add_constant(df_model[list(regressor_params)])
    
    # 2. Estimate the model with Firm and Year Fixed Effects
    mod_ols = PanelOLS(Y, X,
                        entity_effects='i' in mod.fe,
                        time_effects='t' in mod.fe
                    )
    
    # Fit model with firm-clustered standard errors
    res = mod_ols.fit(cov_type='clustered', cluster_entity=True)
    
    # Extract \beta results iterating through full_params
    beta = { p: res.params[p] for p in regressor_params }
    print(f"✅ Model '{model_name}' estimated: {", ".join([f'{k}={v:.3f}' for k, v in beta.items()])}")
    
    # 5. Store the results and parameters
    param_str = ""
    param_str += "Model Parameters:\n"
    for key, value in mod.items():
        if isinstance(value, list) and len(value) == 0:
            continue
        if value is None:
            continue
        param_str += f"{key}: {value}\n"

    output_str = ""
    output_str += f"Model '{model_name}': {mod.description}\n"
    output_str += param_str + "\n"
    output_str += f"{res.summary}\n\n"
    output_str += "=".format(87) + "\n\n"

    return {
        **beta,
        # Safely extract time effects if they exist
        'alpha_i': res.estimated_effects.xs('entity_effects', level=1) if 'entity_effects' in res.estimated_effects.index.names else None,
        'gamma_t': res.estimated_effects.xs('time_effects', level=1) if 'time_effects' in res.estimated_effects.index.names else None, 
        'summary': res.summary,
        'output_str': output_str
    }

# 1a. LMM
$$
\begin{align*}
y_{it} &= \alpha_i + \gamma_t + w_{it}\theta + \beta E[TFP_{-i,g,t}\vert{}g] + \epsilon_{it} \\
x_{it} &=
\begin{pmatrix}
k_{it} & l_{it}
\end{pmatrix}
\end{align*}
$$

In [6]:
import traceback

table_panel_name = "working_yearly_with_tfp_wave"       # Meant to call it "wav" as in weighted average, but accidentally called it wave
table_panel = con.table(table_panel_name)
table_panel_f = table_panel.drop_null(["tfp", "peer_tfp_ttwa_donut", "peer_tfp_pc4_donut", "peer_tfp_pc8"])

# 3. Models: varying Y (gva1 vs gva2) and K (fixed_total vs tangibles)
models = {
    'peer3': {
        'Y': 'tfp',
        'X': ['peer_tfp_ttwa_donut', 'peer_tfp_pc4_donut', 'peer_tfp_pc8'],
        'fe': ['i', 't'],
        'description': 'Base: 3 donut peer TFP effects, firm time fixed effects, no controls'
    },
    'peer3_no_firm_fe': {
        'Y': 'tfp',
        'X': ['peer_tfp_ttwa_donut', 'peer_tfp_pc4_donut', 'peer_tfp_pc8'],
        'fe': ['t'],
        'description': 'Base - firm FE',
        'include': True
    },
    'peer3_employees': {
        'Y': 'tfp',
        'X': ['peer_tfp_ttwa_donut', 'peer_tfp_pc4_donut', 'peer_tfp_pc8'],
        'W_it': ['employees'],
        'fe': ['i', 't'],
        'description': 'Base + employees control',
        'include': True
    }
}

parameter_tables = {}
output_str = ""
latex_str = ""
model_count = 0
for name, mod_obj in models.items():
    try:
        mod = ModelSpec(**mod_obj)
        if not mod.include:
            print(f"⚠️ Model '{name}' is marked as not included. Skipping.")
            continue
        panel_res = run_panel(mod, table_panel_f, name)
        output_str += panel_res['output_str']
        latex_str += panel_res['summary'].as_latex() + "\n\n"
        parameter_tables[name] = panel_res
        model_count += 1
    except Exception as e:
        print(f"❌ Model '{name}' failed. {type(e).__name__}: {e}")
        traceback.print_exc()  # Print the full traceback for debugging

print(f"Panel regressions complete. {model_count} model{'' if model_count == 1 else 's'}, writing.")
with open(dirs.output_dir / "results_1a_llm.txt", "w") as f:
    f.write(output_str)
with open(dirs.output_dir / "results_1a_llm_latex.txt", "w") as f:
    f.write(latex_str)

✅ Model 'peer3' estimated: peer_tfp_ttwa_donut=0.033, peer_tfp_pc4_donut=0.015, peer_tfp_pc8=0.280
✅ Model 'peer3_no_firm_fe' estimated: peer_tfp_ttwa_donut=0.204, peer_tfp_pc4_donut=0.142, peer_tfp_pc8=0.559
✅ Model 'peer3_employees' estimated: ln_employees=0.009, peer_tfp_ttwa_donut=0.033, peer_tfp_pc4_donut=0.015, peer_tfp_pc8=0.280
Panel regressions complete. 3 models, writing.


# 1b. Industries group model

# 2. Distance decay model

$$
\begin{align*}
z_{it} &= \alpha_i + \gamma_t + \rho \sum_{j \neq i} f(d_{ij}) \cdot z_{jt} + \epsilon_{it}   \\
y_{it} &= \alpha_i + \gamma_t + \beta_1 k_{it} + \beta_2 l_{it} + \rho \sum_{j \neq i} w_{ij} z_{jt} + \epsilon_{it}
\end{align*}
$$

In [8]:
import traceback

table_panel_name = "working_yearly_with_tfp_wave"       # Meant to call it "wav" as in weighted average, but accidentally called it wave
table_panel = con.table(table_panel_name)
table_panel_f = table_panel.drop_null(["tfp", "peer_tfp_ttwa_donut", "peer_tfp_pc4_donut", "peer_tfp_pc8"])

# 3. Models: varying Y (gva1 vs gva2) and K (fixed_total vs tangibles)
models = {
    'dd_gravity1': {
        'Y': 'tfp',
        'X': ['tfp_wav1'],
        'W_it': ['nb_peers', 'employees'],
        'fe': ['i', 't'],
        'description': 'Base: 1/d distance peer effect'
    },
    'dd_neg2': {
        'Y': 'tfp',
        'X': ['tfp_wav2'],
        'W_it': ['nb_peers', 'employees'],
        'fe': ['i', 't'],
        'description': 'Base: -1/d^2 distance peer effect'
    },
    'dd_expdd3': {
        'Y': 'tfp',
        'X': ['tfp_wav3'],
        'W_it': ['nb_peers', 'employees'],
        'fe': ['i', 't'],
        'description': 'Base: exp(-d / 1000) distance peer effect'
    }
}

parameter_tables = {}
output_str = ""
model_count = 0
for name, mod_obj in models.items():
    try:
        mod = ModelSpec(**mod_obj)
        if not mod.include:
            print(f"⚠️ Model '{name}' is marked as not included. Skipping.")
            continue
        panel_res = run_panel(mod, table_panel_f, name)
        output_str += panel_res['output_str']
        parameter_tables[name] = panel_res
        model_count += 1
    except Exception as e:
        print(f"❌ Model '{name}' failed. {type(e).__name__}: {e}")
        traceback.print_exc()  # Print the full traceback for debugging

out_file = dirs.output_dir / "results_2_dd.txt"
print(f"Panel regressions complete. {model_count} model{'' if model_count == 1 else 's'}, writing to {out_file}")
with open(out_file, "w") as f:
    f.write(output_str)

✅ Model 'dd_gravity1' estimated: ln_nb_peers=0.027, ln_employees=0.009, tfp_wav1=0.212
✅ Model 'dd_neg2' estimated: ln_nb_peers=0.026, ln_employees=0.009, tfp_wav2=0.174
✅ Model 'dd_expdd3' estimated: ln_nb_peers=0.036, ln_employees=0.009, tfp_wav3=0.288
Panel regressions complete. 3 models, writing.


In [ ]:
# import ibis
# from utils.f_0_dirs import get_data_dirs
# db_path = get_data_dirs().output_dir / "fame_data.duckdb"
# con = ibis.duckdb.connect(db_path)

con.raw_sql("CHECKPOINT;")
con.disconnect()